In [1]:
"""
This protocol creates Bradford working standards, high range (100-1500 ug/mL).
Follows the standard test tube / microplate table in the Thermo Fisher manual:
https://documents.thermofisher.com/TFS-Assets/LSG/manuals/MAN0011181_Coomassie_Bradford_Protein_Asy_UG.pdf

Author : Harley King
Date   : 2026-03-19

PREP:
1) Load 1000 uL BSA stock into deepwell plate well A1.
2) A1 remains vial A after the routine; the script withdraws 700 uL total and leaves 300 uL.

SETUP:
1) Tip carrier at rails 25 with 1000 uL filtered tips at pos[0] and 50 uL filtered tips at pos[1].
2) Trough with diluent at pos[1] on carrier at rails 13.
3) BioER deepwell plate at pos[0] on carrier at rails 19.
4) Standards are created in deepwell wells A1:H1 and H2.
"""


'\nThis protocol creates Bradford working standards, high range (100-1500 ug/mL).\nFollows the standard test tube / microplate table in the Thermo Fisher manual:\nhttps://documents.thermofisher.com/TFS-Assets/LSG/manuals/MAN0011181_Coomassie_Bradford_Protein_Asy_UG.pdf\n\nAuthor : Harley King\nDate   : 2026-03-19\n\nPREP:\n1) Load 1000 uL BSA stock into deepwell plate well A1.\n2) A1 remains vial A after the routine; the script withdraws 700 uL total and leaves 300 uL.\n\nSETUP:\n1) Tip carrier at rails 25 with 1000 uL filtered tips at pos[0] and 50 uL filtered tips at pos[1].\n2) Trough with diluent at pos[1] on carrier at rails 13.\n3) BioER deepwell plate at pos[0] on carrier at rails 19.\n4) Standards are created in deepwell wells A1:H1 and H2.\n'

**Preparation of BSA standards**

This notebook implements only the high-range Bradford standards. `A1` is the stock source and remains vial `A`; vials `B` through `H` are created in `B1:H1`, and vial `I` is created in `H2`.

| Vial | Deepwell well | Volume of diluent | Volume and source of BSA | Final BSA concentration |
| ---- | ------------- | ----------------: | ------------------------ | ----------------------: |
| A | A1 | 0 uL | 300 uL retained stock | 2,000 ug/mL |
| B | B1 | 125 uL | 375 uL of stock from A1 | 1,500 ug/mL |
| C | C1 | 325 uL | 325 uL of stock from A1 | 1,000 ug/mL |
| D | D1 | 175 uL | 175 uL of vial B from B1 | 750 ug/mL |
| E | E1 | 325 uL | 325 uL of vial C from C1 | 500 ug/mL |
| F | F1 | 325 uL | 325 uL of vial E from E1 | 250 ug/mL |
| G | G1 | 325 uL | 325 uL of vial F from F1 | 125 ug/mL |
| H | H1 | 400 uL | 100 uL of vial G from G1 | 25 ug/mL |
| I | H2 | 400 uL | 0 | 0 ug/mL |


In [2]:
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    pass


In [3]:
import asyncio
import random
from copy import deepcopy
from typing import Dict, List

from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.liquid_handling.backends import STARBackend
from pylabrobot.resources import (
    hamilton_96_tiprack_50uL_filter,
    hamilton_96_tiprack_1000uL,
)
from pylabrobot.resources.agenbio.plates import AGenBio_1_troughplate_100000uL_Fl
from pylabrobot.resources.bioer.plates import BioER_96_wellplate_Vb_2200uL
from pylabrobot.resources.hamilton import STARLetDeck, MFX_CAR_L5_base, TIP_CAR_480_A00
from pylabrobot.resources.hamilton.mfx_modules import Hamilton_MFX_plateholder_DWP_metal_tapped

backend = STARBackend()
lh = LiquidHandler(backend=backend, deck=STARLetDeck())
await lh.setup(skip_autoload=True)

tip_car = TIP_CAR_480_A00("tip_car")
lh.deck.assign_child_resource(tip_car, rails=25)
tiprack_1000 = hamilton_96_tiprack_1000uL("tips_00")
tiprack_50 = hamilton_96_tiprack_50uL_filter("tips_01")
tip_car[0] = tiprack_1000
tip_car[1] = tiprack_50

dwp_mod_trough = Hamilton_MFX_plateholder_DWP_metal_tapped("dwp_mod_trough")
car_13 = MFX_CAR_L5_base("car_13", modules={1: dwp_mod_trough})
lh.deck.assign_child_resource(car_13, rails=13)
trough = AGenBio_1_troughplate_100000uL_Fl("diluent_trough")
dwp_mod_trough.assign_child_resource(trough)

dwp_mod_dw = Hamilton_MFX_plateholder_DWP_metal_tapped("dwp_mod_dw")
car_19 = MFX_CAR_L5_base("car_19", modules={0: dwp_mod_dw})
lh.deck.assign_child_resource(car_19, rails=19)
dw_plate = BioER_96_wellplate_Vb_2200uL("dw_plate")
dwp_mod_dw.assign_child_resource(dw_plate)


2026-03-25 13:34:10,295 - pylabrobot.io.usb - INFO - Finding USB device...
2026-03-25 13:34:10,303 - pylabrobot.io.usb - INFO - Found USB device.
2026-03-25 13:34:10,306 - pylabrobot.io.usb - INFO - Found endpoints. 
Write:
       ENDPOINT 0x2: Bulk OUT ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :    0x2 OUT
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0 
Read:
       ENDPOINT 0x81: Bulk IN ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :   0x81 IN
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0
2026-03-25 13:34:13,485 - pylabrobot - INFO - Running backend initialization procedure.
/tmp/ipykernel_364551/4105488656.py:28: DeprecationWarning: Hamilton_MFX_plateholder

In [4]:
await lh.backend.request_name_of_last_faulty_parameter()

{'vp': 'vp', 'id': 15}

In [5]:
CHANNEL = random.randint(0, 7)
USE_CHANNELS = [CHANNEL]
TIP_SEQUENCE = [f"A{i}" for i in range(1, 9)]

BSA_STARTING_VOLUME_UL = 1000
BSA_FINAL_REMAINING_VOLUME_UL = 300
DILUENT_ASPIRATE_HEIGHT_MM = 2
DILUENT_DISPENSE_FLOW_RATE_UL_S = 375

print(f"Using STAR pipetting channel {CHANNEL + 1} for this Bradford standards run.")

STANDARD_MAP: List[Dict[str, object]] = [
    {"vial": "A", "well": "A1", "diluent_ul": 0,   "source": None,  "transfer_ul": 0,   "final_volume_ul": 300, "concentration_ug_ml": 2000},
    {"vial": "B", "well": "B1", "diluent_ul": 125, "source": "A1", "transfer_ul": 375, "final_volume_ul": 500, "concentration_ug_ml": 1500},
    {"vial": "C", "well": "C1", "diluent_ul": 325, "source": "A1", "transfer_ul": 325, "final_volume_ul": 650, "concentration_ug_ml": 1000},
    {"vial": "D", "well": "D1", "diluent_ul": 175, "source": "B1", "transfer_ul": 175, "final_volume_ul": 350, "concentration_ug_ml": 750},
    {"vial": "E", "well": "E1", "diluent_ul": 325, "source": "C1", "transfer_ul": 325, "final_volume_ul": 650, "concentration_ug_ml": 500},
    {"vial": "F", "well": "F1", "diluent_ul": 325, "source": "E1", "transfer_ul": 325, "final_volume_ul": 650, "concentration_ug_ml": 250},
    {"vial": "G", "well": "G1", "diluent_ul": 325, "source": "F1", "transfer_ul": 325, "final_volume_ul": 650, "concentration_ug_ml": 125},
    {"vial": "H", "well": "H1", "diluent_ul": 400, "source": "G1", "transfer_ul": 100, "final_volume_ul": 500, "concentration_ug_ml": 25},
    {"vial": "I", "well": "H2", "diluent_ul": 400, "source": None,  "transfer_ul": 0,   "final_volume_ul": 400, "concentration_ug_ml": 0},
]

PRELOAD_DILUENT_STEPS = [
    {"dest": entry["well"], "volume_ul": entry["diluent_ul"]}
    for entry in STANDARD_MAP
    if entry["diluent_ul"]
]

TRANSFER_STEPS = [
    {"src": entry["source"], "dest": entry["well"], "volume_ul": entry["transfer_ul"]}
    for entry in STANDARD_MAP
    if entry["source"] and entry["transfer_ul"]
]

PREPARED_STANDARD_VOLUMES = {entry["well"]: entry["final_volume_ul"] for entry in STANDARD_MAP}
SOURCE_WELLS = ["A1", "B1", "C1", "E1", "F1", "G1"]

MIX_PLAN = {
    "B1": {"mixvol": 350, "repetitions": 2},
    "C1": {"mixvol": 450, "repetitions": 2},
    "D1": {"mixvol": 250, "repetitions": 2},
    "E1": {"mixvol": 450, "repetitions": 2},
    "F1": {"mixvol": 450, "repetitions": 2},
    "G1": {"mixvol": 450, "repetitions": 2},
    "H1": {"mixvol": 350, "repetitions": 2},
}


def well(name: str):
    return dw_plate[name]


def well_item(name: str):
    return dw_plate.get_item(name)


def tip(name: str):
    return tiprack_1000[name]


def safe_dispense_height(total_after_dispense: float) -> float:
    if total_after_dispense <= 400:
        return 4
    if total_after_dispense <= 500:
        return 5
    return 6


def expected_residual_volumes(starting_stock_volume_ul: int) -> Dict[str, float]:
    residuals = {"A1": starting_stock_volume_ul}
    for well_name, prepared_volume_ul in PREPARED_STANDARD_VOLUMES.items():
        if well_name != "A1":
            residuals[well_name] = prepared_volume_ul
    for step in TRANSFER_STEPS:
        residuals[step["src"]] -= step["volume_ul"]
    return residuals


def validate_standard_plan(starting_stock_volume_ul: int = BSA_STARTING_VOLUME_UL) -> Dict[str, object]:
    if starting_stock_volume_ul < 1000:
        raise ValueError("A1 must start with at least 1000 uL to leave 300 uL after supplying B and C.")

    expected_residuals = expected_residual_volumes(starting_stock_volume_ul)

    volumes = {"A1": starting_stock_volume_ul}
    for step in PRELOAD_DILUENT_STEPS:
        volumes[step["dest"]] = volumes.get(step["dest"], 0) + step["volume_ul"]

    for step in TRANSFER_STEPS:
        src = step["src"]
        dest = step["dest"]
        vol = step["volume_ul"]
        if vol > 1000:
            raise ValueError(f"Transfer {src} -> {dest} exceeds 1000 uL: {vol} uL")
        if volumes.get(src, 0) < vol:
            raise ValueError(f"Source {src} does not have enough volume for {dest}: {volumes.get(src, 0)} uL left, need {vol} uL")
        volumes[src] -= vol
        volumes[dest] = volumes.get(dest, 0) + vol

    if volumes["A1"] < BSA_FINAL_REMAINING_VOLUME_UL:
        raise ValueError(f"A1 ends below the required retained stock volume: {volumes['A1']} uL")

    for well_name, expected in expected_residuals.items():
        actual = volumes.get(well_name, 0)
        if actual != expected:
            raise ValueError(f"{well_name} residual volume mismatch: expected {expected} uL, found {actual} uL")

    for well_name, mix_cfg in MIX_PLAN.items():
        if mix_cfg["mixvol"] >= PREPARED_STANDARD_VOLUMES[well_name]:
            raise ValueError(f"Mix volume for {well_name} must stay below the well volume.")

    residuals = {well_name: volumes[well_name] for well_name in SOURCE_WELLS}
    return {
        "starting_stock_volume_ul": starting_stock_volume_ul,
        "prepared_standard_volumes_ul": deepcopy(PREPARED_STANDARD_VOLUMES),
        "residual_volumes_ul": {well_name: volumes[well_name] for well_name in expected_residuals},
        "source_residuals_ul": residuals,
    }


async def _pickup_1000(tip_name: str):
    await lh.pick_up_tips(tip(tip_name), use_channels=USE_CHANNELS)


async def _drop_1000(tip_name: str):
    await lh.drop_tips(tip(tip_name), use_channels=USE_CHANNELS)


async def aspirate_diluent(volume_ul: float):
    await lh.aspirate(
        trough["A1"],
        vols=[volume_ul],
        use_channels=USE_CHANNELS,
        liquid_height=[DILUENT_ASPIRATE_HEIGHT_MM],
        flow_rates=[400],
    )


async def aspirate_from_well(src_name: str, volume_ul: float, remaining_volume_ul: float):
    src = well(src_name)
    src_item = well_item(src_name)
    average_volume_during_aspiration = max(remaining_volume_ul - (volume_ul / 2), 1)
    liquid_height = max(src_item.compute_height_from_volume(average_volume_during_aspiration) - 1, 1)
    await lh.aspirate(
        src,
        vols=[volume_ul],
        use_channels=USE_CHANNELS,
        # liquid_height=[liquid_height],
        liquid_height=[4],
        flow_rates=[400],
        swap_speed=[160],
    )


async def dispense_to_well(dest_name: str, volume_ul: float, total_after_dispense_ul: float):
    await lh.dispense(
        well(dest_name),
        vols=[volume_ul],
        use_channels=USE_CHANNELS,
        liquid_height=[safe_dispense_height(total_after_dispense_ul)],
        flow_rates=[500],
        blow_out=[1],
        swap_speed=[160],
        settling_time=[1],
    )


async def dispense_diluent_to_well(dest_name: str, volume_ul: float, total_after_dispense_ul: float):
    await lh.dispense(
        well(dest_name),
        vols=[volume_ul],
        use_channels=USE_CHANNELS,
        liquid_height=[safe_dispense_height(total_after_dispense_ul)],
        flow_rates=[DILUENT_DISPENSE_FLOW_RATE_UL_S],
        blow_out=[1],
        swap_speed=[160],
        settling_time=[1],
    )


async def mix_well(dest_name: str, mixvol: float, totalvol: float, repetitions: int = 2):
    target = well(dest_name)
    aspirate_height = 6 if totalvol >= 500 else 4
    dispense_height = 4 if totalvol <= 500 else 6
    for _ in range(repetitions):
        await lh.aspirate(
            target,
            vols=[mixvol],
            use_channels=USE_CHANNELS,
            # liquid_height=[aspirate_height],
            liquid_height=[4],
            flow_rates=[400],
            swap_speed=[160],
        )
        await lh.dispense(
            target,
            vols=[mixvol],
            use_channels=USE_CHANNELS,
            liquid_height=[dispense_height],
            flow_rates=[500],
            blow_out=[1],
            swap_speed=[160],
            settling_time=[1],
        )


async def preload_diluent(current_volumes: Dict[str, float]):
    await _pickup_1000(TIP_SEQUENCE[0])
    for step in PRELOAD_DILUENT_STEPS:
        dest = step["dest"]
        vol = step["volume_ul"]
        await aspirate_diluent(vol)
        total_after = current_volumes.get(dest, 0) + vol
        await dispense_diluent_to_well(dest, vol, total_after)
        current_volumes[dest] = total_after
    await _drop_1000(TIP_SEQUENCE[0])


async def transfer_and_mix(src: str, dest: str, vol: float, tip_name: str, current_volumes: Dict[str, float]):
    await _pickup_1000(tip_name)
    await aspirate_from_well(src, vol, current_volumes[src])
    current_volumes[src] -= vol

    total_after = current_volumes.get(dest, 0) + vol
    await dispense_to_well(dest, vol, total_after)
    current_volumes[dest] = total_after

    mix_cfg = MIX_PLAN.get(dest)
    if mix_cfg is not None:
        await mix_well(dest, mix_cfg["mixvol"], current_volumes[dest], repetitions=mix_cfg["repetitions"])

    await _drop_1000(tip_name)


async def create_bradford_standards_hi_range(starting_stock_volume_ul: int = BSA_STARTING_VOLUME_UL):
    plan = validate_standard_plan(starting_stock_volume_ul=starting_stock_volume_ul)
    current_volumes = {"A1": starting_stock_volume_ul}

    await preload_diluent(current_volumes)

    for tip_name, step in zip(TIP_SEQUENCE[1:], TRANSFER_STEPS):
        await transfer_and_mix(
            src=step["src"],
            dest=step["dest"],
            vol=step["volume_ul"],
            tip_name=tip_name,
            current_volumes=current_volumes,
        )

    final_plan = validate_standard_plan(starting_stock_volume_ul=starting_stock_volume_ul)
    if current_volumes != final_plan["residual_volumes_ul"]:
        raise RuntimeError(f"Tracked volumes do not match validation result: {current_volumes} vs {final_plan['residual_volumes_ul']}")

    return {
        "prepared_standard_volumes_ul": deepcopy(PREPARED_STANDARD_VOLUMES),
        "residual_volumes_ul": deepcopy(current_volumes),
        "source_residuals_ul": final_plan["source_residuals_ul"],
        "standard_map": deepcopy(STANDARD_MAP),
    }


Using STAR pipetting channel 5 for this Bradford standards run.


In [6]:
plan_check = validate_standard_plan()
plan_check

# Run on the instrument when ready:
result = await create_bradford_standards_hi_range()
result

# Shutdown when finished:
# await lh.stop()


{'prepared_standard_volumes_ul': {'A1': 300,
  'B1': 500,
  'C1': 650,
  'D1': 350,
  'E1': 650,
  'F1': 650,
  'G1': 650,
  'H1': 500,
  'H2': 400},
 'residual_volumes_ul': {'A1': 300,
  'B1': 325,
  'C1': 325,
  'D1': 350,
  'E1': 325,
  'F1': 325,
  'G1': 550,
  'H1': 500,
  'H2': 400},
 'source_residuals_ul': {'A1': 300,
  'B1': 325,
  'C1': 325,
  'E1': 325,
  'F1': 325,
  'G1': 550},
 'standard_map': [{'vial': 'A',
   'well': 'A1',
   'diluent_ul': 0,
   'source': None,
   'transfer_ul': 0,
   'final_volume_ul': 300,
   'concentration_ug_ml': 2000},
  {'vial': 'B',
   'well': 'B1',
   'diluent_ul': 125,
   'source': 'A1',
   'transfer_ul': 375,
   'final_volume_ul': 500,
   'concentration_ug_ml': 1500},
  {'vial': 'C',
   'well': 'C1',
   'diluent_ul': 325,
   'source': 'A1',
   'transfer_ul': 325,
   'final_volume_ul': 650,
   'concentration_ug_ml': 1000},
  {'vial': 'D',
   'well': 'D1',
   'diluent_ul': 175,
   'source': 'B1',
   'transfer_ul': 175,
   'final_volume_ul': 350,
